In [0]:
spark.conf.set(
    "fs.azure.account.key.datalake0501.dfs.core.windows.net",
    "YOUR_STORAGE_ACCOUNT_KEY_HERE"  # Replace with your key
)

# Read from Bronze
df_bronze = spark.read.csv(
    'abfss://bronze@datalake0501.dfs.core.windows.net/ecommerce/raw/ecommerce_dataset_1m.csv',
    header=True,
    inferSchema=True
)

print(f"Bronze records: {df_bronze.count()}")
df_bronze.printSchema()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

df_silver = df_bronze \
    .dropDuplicates(['order_id']) \
    .filter(F.col('order_id').isNotNull()) \
    .filter(F.col('customer_id').isNotNull()) \
    .filter(F.col('total_price_usd') > 0) \
    .withColumn('is_weekend', F.when(F.col('is_weekend') == 'Yes', True).otherwise(False)) \
    .withColumn('coupon_used', F.when(F.col('coupon_used') == 'Yes', True).otherwise(False)) \
    .withColumn('installment_plan', F.when(F.col('installment_plan') == 'Yes', True).otherwise(False)) \
    .withColumn('abandoned_cart_before', F.when(F.col('abandoned_cart_before') == 'Yes', True).otherwise(False)) \
    .withColumn('support_ticket_created', F.when(F.col('support_ticket_created') == 'Yes', True).otherwise(False)) \
    .withColumn('unit_price_usd', F.round(F.col('unit_price_usd').cast(DecimalType(10,2)), 2)) \
    .withColumn('profit_usd', F.round(F.col('profit_usd').cast(DecimalType(10,2)), 2)) \
    .withColumn('total_price_usd', F.round(F.col('total_price_usd').cast(DecimalType(10,2)), 2)) \
    .withColumn('profit_margin_percent', F.round(F.col('profit_margin_percent').cast(DecimalType(5,2)), 2)) \
    .withColumn('return_reason', F.when(F.col('return_reason').isNull(), 'No Return').otherwise(F.col('return_reason'))) \
    .withColumn('customer_feedback', F.when(F.col('customer_feedback').isNull(), 'No Feedback').otherwise(F.col('customer_feedback'))) \
    .withColumn('ingestion_timestamp', F.current_timestamp())

print(f"Silver records: {df_silver.count()}")
df_silver.show(3)

In [0]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .save("abfss://silver@datalake0501.dfs.core.windows.net/ecommerce/orders/")

print("Silver Delta table written successfully!")

In [0]:
# Verify Silver Delta table
df_check = spark.read.format("delta").load(
    "abfss://silver@datalake0501.dfs.core.windows.net/ecommerce/orders/"
)

print(f"Silver records: {df_check.count()}")
print(f"Columns: {len(df_check.columns)}")

# Check new business columns exist
df_check.select(
    'order_id', 'order_year', 'order_month',
    'profit_margin_percent', 'is_weekend', 'coupon_used',
    'ingestion_timestamp'
).show(3)

In [0]:
df_silver_enhanced.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("order_year", "order_month") \
    .save("abfss://silver@datalake0501.dfs.core.windows.net/ecommerce/orders/")

print("Orders Delta table written!")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

# Fresh read from Bronze
df_base = spark.read.csv(
    'abfss://bronze@datalake0501.dfs.core.windows.net/ecommerce/raw/ecommerce_dataset_1m.csv',
    header=True,
    inferSchema=True
)

# Customers dimension
df_customers = df_base.select(
    'customer_id', 'customer_name', 'gender', 'age',
    'customer_segment', 'country', 'city',
    'customer_loyalty_score', 'total_orders_by_customer',
    'account_creation_date'
).dropDuplicates(['customer_id']) \
 .filter(F.col('customer_id').isNotNull())

df_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("abfss://silver@datalake0501.dfs.core.windows.net/ecommerce/customers/")

print(f"Customers written: {df_customers.count()}")

# Products dimension
df_products = df_base.select(
    'product_id', 'product_name', 'category', 'sub_category',
    'brand', 'product_rating_avg', 'product_reviews_count',
    'stock_quantity', 'unit_price_usd'
).dropDuplicates(['product_id']) \
 .filter(F.col('product_id').isNotNull())

df_products.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("abfss://silver@datalake0501.dfs.core.windows.net/ecommerce/products/")

print(f"Products written: {df_products.count()}")

In [0]:
silver_orders = spark.read.format("delta").load(
    "/mnt/silver/orders"
)